In [28]:
import pandas as pd
import ipaddress
import spacy
import time


In [30]:
nlp = spacy.load("en_core_web_sm")

In [31]:
df = pd.read_csv("/content/drive/MyDrive/search-results-2024-11-07T01_42_33.367-0800.csv")

In [41]:
def extract_username(email):
    return email.split('@')[0]

In [46]:
def generate_ngrams(text, n_range=(3, 6)):
    ngrams = []
    length = len(text)
    for n in range(n_range[0], min(n_range[1]+1, length+1)):
        ngrams.extend([text[i:i+n] for i in range(length-n+1)])
    return ngrams

In [44]:
def contains_person_name_from_ngrams(ngrams):
    for token in ngrams:
        doc = nlp(token)
        for ent in doc.ents:
            if ent.label_ == 'PERSON':
                return 1
    return 0

In [35]:
def is_valid_ip(ip):
    if not ip:
        return -1
    try:
        ipaddress.ip_address(ip)
        return 1
    except ValueError:
        return 0

In [47]:
def label_data(row):
    # Username-based features
    username = extract_username(row['user_id'])
    length_of_username = len(username)
    num_special_chars = sum(not char.isalnum() for char in username)

    # N-grams and person name check
    ngrams = generate_ngrams(username, n_range=(3, 6))
    has_person_name_ngram = contains_person_name_from_ngrams(ngrams)

    #IP features
    ip_valid = is_valid_ip(row['source_ip'])

    if ip_valid == 0:
        return 0
    if has_person_name_ngram != 1:
        return 0
    if length_of_username > 20 or num_special_chars > 3:
        return 0

    return 1


Accuracy: 0.354
Precision: 0.34899396161894725
Recall: 0.3566006817077352
F1 Score: 0.34767900179448896
Confusion Matrix:
[[28  1  2  8  1  3  7 17 30 14]
 [ 6 47  0  0 13 23  1  0  5 10]
 [ 6  2 29 15 14  3 16  7  7  6]
 [ 1  0  7 52  1  0  6 14  3  1]
 [ 5 10  4  4 41  9  8  0  3 10]
 [ 3 27  4  1 14 29  6  0  6  8]
 [ 5  3 18 10  9  3 15  7 17 11]
 [ 8  2  5 20  1  0  5 63  4  4]
 [13  3  3  8  2  2  5  8 24 13]
 [ 4  7  6 10 13  8 17  5 15 26]]
Now capture real-time keystroke data for prediction.


KeyboardInterrupt: 

In [ ]:
df['label'] = df.apply(label_data, axis=1)

In [ ]:
output_file = "training_data.csv"
df.to_csv(output_file, index=False)
print(f"Data labeling complete. Labeled data saved to '{output_file}'.")

In [ ]:
df['label'].value_counts()

In [ ]:
# Main function
def main():
    keystrokes = collect_keystroke_data(duration=10)  # Capture keystrokes for 10 seconds
    features = extract_features(keystrokes)
    # print(features)

    # Assume two users for demonstration
    labels = ['user1', 'user2']
    user_labels = np.random.choice(labels, len(features))  # Simulated user labels
    # print(user_labels)

    model = train_model(features, user_labels)

    # Simulated authentication using the trained model
    # Replace with actual keystroke data from a user
    user_keystroke_features = [{KEY_HOLD: 0, KEY_RELEASE: 0.05, TYPING_SPEED: 9.13}]
    user = authenticate_user(model, user_keystroke_features)
    # print(user)
    print("Authenticated User:", labels[user])

if __name__ == "__main__":
    main()